# Inspect interleaved data and generate speech

Reconstruct an aligned speech–text sample, inspect its interleaved training representation, and generate speech with an LF²AR checkpoint.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, Markdown, display

from interleaved_lm import PerceptionExpressionAdaptedTextLM
from interleaved_lm.audio import decode_hubert, generate_speech, load_codehifigan
from interleaved_speech_data import PackedShard


In [ ]:
MODEL_ID = os.environ.get("INTERLEAVED_MODEL_ID", "tiagoCuervo/lf2ar-speech-360m")
DATA_ROOT = Path(os.environ.get("INTERLEAVED_DATA_ROOT", "data"))
DATASET = os.environ.get("INTERLEAVED_DATASET", "tinystories-kokoro")
SPEECH_TOKENS = "hubert"
SPLIT = "train"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = PerceptionExpressionAdaptedTextLM.from_pretrained(MODEL_ID, device=device).eval()
vocoder = load_codehifigan(device=device, vocab_size=500)


## Aligned utterance

The audio is reconstructed from the stored speech units. Word boundaries come from the aligned end-of-word markers.


In [ ]:
meta_paths = sorted((DATA_ROOT / DATASET / SPEECH_TOKENS / SPLIT).glob("*/meta.json"))
if not meta_paths:
    raise FileNotFoundError("prepare at least one canonical aligned shard first")

shard = PackedShard(meta_paths[0].parent, verify_checksums=True)
if shard.meta["profile"] != "aligned":
    raise ValueError("this notebook requires an aligned shard")

paired = shard.sample(0)
prep = shard.meta["preprocessing"]
pad_id = prep["text_pad_id"]
eow_id = prep["text_end_of_word_id"]
eos_id = prep["text_eos_id"]

words = []
word_tokens = []
word_audio_start = 0
audio_position = 0
for token, duration in zip(paired["text"], paired["durations"]):
    token = int(token)
    audio_position += int(duration)
    if token == eow_id:
        if word_tokens:
            words.append((word_tokens, word_audio_start, audio_position))
        word_tokens = []
        word_audio_start = audio_position
    elif token not in {pad_id, eos_id}:
        word_tokens.append(token)

def decode_words(selected):
    return " ".join(
        model.global_workspace.txt_tokenizer.decode(tokens).strip()
        for tokens, _, _ in selected
    )

display(Markdown("**Aligned text:** " + decode_words(words)))
waveform = decode_hubert(np.array(paired["audio"], copy=True), vocoder)
display(Audio(waveform, rate=16_000))


## Interleaved training sample

Blue spans are text, orange spans are speech, and gray positions switch modality. Speech chunks are playable below the overview.


In [ ]:
text_chunk_words = 10
speech_chunk_words = 6
chunks = []
position = 0
modality = "text"
while position < len(words):
    width = text_chunk_words if modality == "text" else speech_chunk_words
    end = min(position + width, len(words))
    chunks.append((modality, position, end))
    position = end
    modality = "speech" if modality == "text" else "text"

labels = []
for index, (modality, start, end) in enumerate(chunks):
    if index:
        labels.append(2)
    if modality == "text":
        labels.extend([0] * sum(len(tokens) for tokens, _, _ in words[start:end]))
    else:
        labels.extend([1] * (words[end - 1][2] - words[start][1]))

colors = plt.matplotlib.colors.ListedColormap(["#4c78a8", "#f58518", "#9d9da1"])
plt.figure(figsize=(14, 1.4))
plt.imshow(np.asarray(labels)[None], aspect="auto", interpolation="nearest", cmap=colors)
plt.yticks([])
plt.xlabel("sequence step")
plt.title("text / speech / switch")
plt.show()

for index, (modality, start, end) in enumerate(chunks):
    if index:
        display(Markdown("--- *switch modality* ---"))
    if modality == "text":
        display(Markdown("**TEXT:** " + decode_words(words[start:end])))
    else:
        display(Markdown("**SPEECH:**"))
        audio_start = words[start][1]
        audio_end = words[end - 1][2]
        codes = np.array(paired["audio"][audio_start:audio_end], copy=True)
        display(Audio(decode_hubert(codes, vocoder), rate=16_000))


## Generate speech from text


In [ ]:
torch.manual_seed(7)
units = generate_speech(
    model,
    "Once upon a time, a small robot found a garden.",
    max_new_tokens=300,
    temperature=0.7,
    top_k=50,
)
generated_waveform = decode_hubert(units, vocoder, vocab_size=500)
sf.write("lf2ar_generation.wav", generated_waveform, 16_000)
display(Audio(generated_waveform, rate=16_000))
